# Module 1 - WER Validation

This notebook compares `faster-whisper` `base` vs `small` on `edinburghcstr/ami` (`ihm`) test utterances.

Important evaluation notes:
- Raw utterance-level WER is kept for completeness.
- We also compute normalized utterance-level WER to reduce formatting noise.
- We compute corpus-level WER so tiny utterances do not dominate the aggregate score.
- We report a filtered view for utterances with at least 3 reference words and at least 1.0s duration.

In [1]:
import os
import re
import sys
import tempfile
from pathlib import Path

import pandas as pd
import soundfile as sf
from datasets import load_dataset
from jiwer import wer

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'Notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.app.module1 import asr as asr_module

In [2]:
SAMPLE_SIZE = 250
MIN_FILTERED_WORDS = 3
MIN_FILTERED_DURATION = 1.0

ds = load_dataset('edinburghcstr/ami', 'ihm', split='test')
sample = ds.shuffle(seed=42).select(range(SAMPLE_SIZE))
sample

Resolving data files:   0%|          | 0/42 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/42 [00:00<?, ?it/s]

Dataset({
    features: ['meeting_id', 'audio_id', 'text', 'audio', 'begin_time', 'end_time', 'microphone_id', 'speaker_id'],
    num_rows: 250
})

In [3]:
def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"\b([a-z])\.\s", r"\1 ", text)
    text = re.sub(r"[^a-z0-9'\s]", ' ', text)
    text = re.sub(r"\s+", ' ', text)
    return text.strip()


def transcribe_row(model_size: str, row) -> str:
    os.environ['WHISPER_MODEL_SIZE'] = model_size
    asr_module._get_whisper_model.cache_clear()
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
        sf.write(tmp.name, row['audio']['array'], row['audio']['sampling_rate'])
        result = asr_module.transcribe_audio(tmp.name, chunk_start=float(row['begin_time']))
    Path(tmp.name).unlink(missing_ok=True)
    return ' '.join(segment.text for segment in result.segments).strip()


def safe_wer(reference: str, prediction: str) -> float:
    if not reference.strip() and not prediction.strip():
        return 0.0
    if not reference.strip():
        return 1.0 if prediction.strip() else 0.0
    return wer(reference, prediction)


def corpus_wer(frame: pd.DataFrame, reference_col: str, prediction_col: str) -> float:
    reference = ' '.join(frame[reference_col].fillna('').astype(str).tolist()).strip()
    prediction = ' '.join(frame[prediction_col].fillna('').astype(str).tolist()).strip()
    return safe_wer(reference, prediction)


def evaluate_model(model_size: str) -> pd.DataFrame:
    rows = []
    for row in sample:
        reference = row['text'].strip()
        prediction = transcribe_row(model_size, row)
        normalized_reference = normalize_text(reference)
        normalized_prediction = normalize_text(prediction)
        duration_sec = float(row['end_time'] - row['begin_time'])
        reference_word_count = len(normalized_reference.split())
        rows.append({
            'model': model_size,
            'audio_id': row['audio_id'],
            'reference': reference,
            'prediction': prediction,
            'reference_norm': normalized_reference,
            'prediction_norm': normalized_prediction,
            'duration_sec': duration_sec,
            'reference_word_count': reference_word_count,
            'raw_utterance_wer': safe_wer(reference, prediction),
            'normalized_utterance_wer': safe_wer(normalized_reference, normalized_prediction),
        })
    return pd.DataFrame(rows)

In [4]:
base_df = evaluate_model('base')
small_df = evaluate_model('small')
results = pd.concat([base_df, small_df], ignore_index=True)

filtered_results = results[
    (results['reference_word_count'] >= MIN_FILTERED_WORDS)
    & (results['duration_sec'] >= MIN_FILTERED_DURATION)
].copy()

summary_rows = []
for model_name, model_df in results.groupby('model'):
    filtered_df = filtered_results[filtered_results['model'] == model_name]
    summary_rows.append({
        'model': model_name,
        'raw_mean_utterance_wer': model_df['raw_utterance_wer'].mean(),
        'normalized_mean_utterance_wer': model_df['normalized_utterance_wer'].mean(),
        'normalized_median_utterance_wer': model_df['normalized_utterance_wer'].median(),
        'normalized_std_utterance_wer': model_df['normalized_utterance_wer'].std(),
        'corpus_wer': corpus_wer(model_df, 'reference_norm', 'prediction_norm'),
        'filtered_mean_wer': filtered_df['normalized_utterance_wer'].mean() if not filtered_df.empty else None,
        'filtered_corpus_wer': corpus_wer(filtered_df, 'reference_norm', 'prediction_norm') if not filtered_df.empty else None,
        'num_rows': len(model_df),
        'num_filtered_rows': len(filtered_df),
    })

summary = pd.DataFrame(summary_rows).sort_values('model').reset_index(drop=True)
summary

,model,raw_mean_utterance_wer,normalized_mean_utterance_wer,normalized_median_utterance_wer,normalized_std_utterance_wer,corpus_wer,filtered_mean_wer,filtered_corpus_wer,num_rows,num_filtered_rows
0,base,1.100883,0.521871,0.322917,0.674417,0.275880,0.294861,0.238630,250,136
1,small,1.026540,0.404435,0.245000,0.439867,0.229456,0.233766,0.202008,250,136


In [5]:
results['duration_bucket'] = pd.cut(
    results['duration_sec'],
    bins=[0, 0.75, 1.5, 3, 6, 15, 60],
    include_lowest=True,
)
results['word_bucket'] = pd.cut(
    results['reference_word_count'],
    bins=[0, 1, 2, 4, 8, 16, 100],
    include_lowest=True,
)

duration_analysis = (
    results.groupby(['model', 'duration_bucket'])['normalized_utterance_wer']
    .mean()
    .reset_index()
)
word_analysis = (
    results.groupby(['model', 'word_bucket'])['normalized_utterance_wer']
    .mean()
    .reset_index()
)

display(duration_analysis)
display(word_analysis)

,model,duration_bucket,normalized_utterance_wer
0,base,"(-0.001, 0.75]",0.781893
1,base,"(0.75, 1.5]",0.478049
2,base,"(1.5, 3.0]",0.527530
3,base,"(3.0, 6.0]",0.285917
4,base,"(6.0, 15.0]",0.220844
5,small,"(-0.001, 0.75]",0.703704
6,small,"(0.75, 1.5]",0.368583
7,small,"(1.5, 3.0]",0.277330
8,small,"(3.0, 6.0]",0.185273
9,small,"(6.0, 15.0]",0.205074


,model,word_bucket,normalized_utterance_wer
0,base,"(-0.001, 1.0]",0.875000
1,base,"(1.0, 2.0]",0.793103
2,base,"(2.0, 4.0]",0.449405
3,base,"(4.0, 8.0]",0.360250
4,base,"(8.0, 16.0]",0.212329
5,base,"(16.0, 100.0]",0.215694
6,small,"(-0.001, 1.0]",0.694444
7,small,"(1.0, 2.0]",0.534483
8,small,"(2.0, 4.0]",0.375000
9,small,"(4.0, 8.0]",0.252236


In [6]:
worst_cases = (
    results.sort_values(['model', 'normalized_utterance_wer', 'duration_sec'], ascending=[True, False, True])
    .groupby('model')
    .head(10)
    [[
        'model',
        'audio_id',
        'duration_sec',
        'reference_word_count',
        'reference',
        'prediction',
        'normalized_utterance_wer',
    ]]
)
worst_cases

,model,audio_id,duration_sec,reference_word_count,reference,prediction,normalized_utterance_wer
121,base,AMI_EN2002d_H00_FEO070_0068205_0068504,2.989990,1,YEAH,"So it's fine, we can do it. Yeah.",7.000000
135,base,AMI_EN2002d_H01_FEO072_0137207_0137686,4.790039,2,OH DEAR,I don't know how to do that.,3.500000
247,base,AMI_IS1009c_H00_FIE088_0163060_0163175,1.150024,1,BATTERY,battery. I think battery.,3.000000
77,base,AMI_EN2002a_H03_MEE071_0051126_0051364,2.380005,1,NO,"No, actually, very much.",3.000000
120,base,AMI_EN2002c_H02_MEE071_0038361_0038476,1.150024,5,SOME ARE QUITE AMUSING ACTUALLY,So I'm going to go out and be using it actually.,2.000000
189,base,AMI_IS1009a_H03_FIO089_0040286_0040648,3.620026,4,UM IS FIFTY MM,And it's 50. I think of course.,1.750000
155,base,AMI_EN2002d_H03_MEE073_0099427_0099484,0.570007,2,OKAY RIGHT,Don't care me.,1.500000
187,base,AMI_IS1009b_H01_FIO087_0026925_0026998,0.730011,2,WHAT FEATURES,Let's check it.,1.500000
138,base,AMI_TS3003d_H00_MTD009PM_0076864_0077108,2.440002,2,POLISH SUPPLIER,Or I'll cry.,1.500000
197,base,AMI_IS1009d_H02_FIO084_0133822_0134090,2.680054,6,WE HAVE UM FOUR EUROS YEAH,We'll be having a full year of C.M.,1.500000
